[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/08-ai-engineering/04_reasoning.ipynb)

*AI Engineering · Notebook 4 of 10*

# Reasoning Models & Test-Time Compute

Models that think before answering: what thinking tokens cost, effort levels, when reasoning helps, self-consistency voting, best-of-N with verifiers, pass@k, process vs outcome checks, routing easy and hard questions, and checking a model's working.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A reasoning model shows its working before the answer — worth it for hard sums, wasteful for 'what's 2 + 2'.

**What interviewers really test here:** Thinking cost and latency, self-consistency vs best-of-N, pass@k, process vs outcome checks, and routing easy vs hard questions.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. What a reasoning model does differently *(Basics)*
- [ ] 2. Choosing how much to think *(Core)*
- [ ] 3. Sampling more: self-consistency and best-of-N *(Core)*
- [ ] 4. Evaluating reasoning: pass@k and checking the working *(Core)*
- [ ] 5. Search over reasoning *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [ ]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
import numpy as np
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Ways to spend test-time compute

| Method | How | Needs |
| --- | --- | --- |
| Thinking | longer internal reasoning | a reasoning model |
| Chain-of-thought prompt | ask a standard model to reason first | nothing extra |
| Self-consistency | sample n, majority vote | answers you can compare |
| Best-of-N | sample n, verifier picks | a verifier (tests, checker, reward model) |
| Search (beam, tree) | expand and prune steps | a step-level verifier |
| Refinement | draft, critique, revise | clear criteria |

### When reasoning helps

- Helps: multi-step maths, code, planning, logic puzzles, analysing long documents with many constraints, agentic tool use with many decisions.
- Doesn't help much: fact lookup, simple classification, extraction, short rewrites, chit-chat.
- Can hurt: latency-critical UIs, very high volume cheap tasks.

---
# Part 2: Lessons, step by step

## Lesson 1: What a reasoning model does differently

*Level: Basics*

A **reasoning** (or "thinking") model writes out a chain of working before its final answer — trying approaches, checking steps, backtracking. Spending more computation at answer time (**test-time compute**) makes it much better at maths, code, planning and tricky multi-step questions.



The trade-offs:




- **Cost**: thinking tokens are billed as output tokens, often many thousands per answer.
- **Latency**: the user waits while it thinks — stream the response, or use thinking only where it helps.
- **No gain on easy tasks**: lookups, classification and simple rewrites don't need it.



```python
reply = client.messages.create(
    model="claude-opus-5-5", max_tokens=16000,
    thinking={"type": "adaptive"},              # the model decides how much to think
    messages=[{"role": "user", "content": problem}])
answer = "".join(b.text for b in reply.content if b.type == "text")
```



APIs differ in the controls they offer (adaptive thinking, effort levels, explicit budgets on older models) — check your model's docs. The thinking comes back as separate blocks you can log for debugging.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 2: Choosing how much to think

*Level: Core*

Treat thinking effort like any other hyper-parameter: **measure accuracy and cost on your own evaluation set** at each level and pick the cheapest that meets the bar. Returns diminish: going from no thinking to a little often helps a lot; going from a lot to the maximum rarely does.



Even better, **route**: send easy questions to a fast model and hard ones to a reasoning model. A small classifier (or the fast model itself) can estimate difficulty. Compare options by **correct answers per dollar**, not accuracy alone.

**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q13](#q13), [Q8](#q8), [Q18](#q18), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 3: Sampling more: self-consistency and best-of-N

*Level: Core*

You can also spend test-time compute by **asking several times**:




- **Self-consistency**: sample n answers and take the majority. If each is right with probability p &gt; 0.5, the majority is right more often than any single answer — and it improves with n. If p &lt; 0.5, voting makes things *worse*.
- **Best-of-N**: sample n and let a **verifier** pick (unit tests for code, a calculator for maths, a reward model). With a perfect verifier you only need one correct sample.



Normalise free-form answers before voting, and use agreement as a confidence signal: if samples disagree, abstain or escalate. Cache the shared prompt so n samples don't cost n times the input.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [ ]:
import matplotlib.pyplot as plt


def voting_curve():
    ns = list(range(1, 26, 2))
    fig, ax = plt.subplots(figsize=(5, 2.6))
    for p in (0.4, 0.6, 0.75):
        acc = [sum(math.comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n // 2 + 1, n + 1)) for n in ns]
        ax.plot(ns, acc, marker="o", label=f"single-sample accuracy {p}")
    ax.set_xlabel("samples voted"); ax.set_ylabel("majority accuracy"); ax.legend(fontsize=7)


voting_curve()

**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q3](#q3), [Q19](#q19), [Q16](#q16), [Q7](#q7), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 4: Evaluating reasoning: pass@k and checking the working

*Level: Core*

For code, run the tests: **pass@k** is the probability that at least one of k samples passes. Estimate it from n ≥ k samples with the unbiased formula 1 − C(n−c, k)/C(n, k), rather than literally taking k samples.



Don't trust the final answer alone. **Process** checks score each step (a chain is only as strong as its weakest step); **outcome** checks score only the final answer. Simple tools catch a lot: recompute every arithmetic line, run the code, check units. Ask for answers in a fixed format (`Answer: …` or `<answer>` tags) so you can parse them reliably.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q4](#q4), [Q15](#q15), [Q14](#q14), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 5: Search over reasoning

*Level: Advanced*

Beyond voting, systems can **search**: generate several candidate next steps, score them with a verifier, keep the best few (beam search), and continue — or explore a tree of thoughts. This trades a lot more compute for accuracy on hard problems with a reliable verifier, such as maths proofs or code with tests.



Reasoning models are trained with **reinforcement learning on verifiable rewards**: problems whose answers can be checked automatically (maths, code), rewarding correct final answers so the model learns useful thinking habits.

**✏️ Your turn:** you now have what you need for [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [ ]:
import re

from itertools import product

from collections import Counter


<a id="q1"></a>
### Q1. What Does Thinking Cost?

🟢 Warm-up · *Thinking is billed as output*

A call reads `inp` input tokens and writes `think` thinking tokens plus `out` answer tokens. Prices are per million tokens. Return `(cost without thinking, cost with thinking)` in dollars, rounded to 5.

**Examples**

- `inp = 3000, think = 8000, out = 400, in_price = 3.0, out_price = 15.0` → `(0.015, 0.135)`
- `inp = 3000, think = 0, out = 400, in_price = 3.0, out_price = 15.0` → `(0.015, 0.015)`

In [ ]:
def thinking_cost(inp, think, out, in_price, out_price):
    """
    A call reads inp input tokens and writes think thinking tokens plus out
    answer tokens. Prices are per million tokens. Return (cost without thinking,
    cost with thinking) in dollars, rounded to 5.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(thinking_cost, [
    ((3000, 8000, 400, 3.0, 15.0), (0.015, 0.135)),
    ((3000, 0, 400, 3.0, 15.0), (0.015, 0.015)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute each part step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two expressions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute each part step by step.

```python
def thinking_cost(inp, think, out, in_price, out_price):
    input_cost = inp * in_price / 1_000_000
    answer_cost = out * out_price / 1_000_000
    thinking = think * out_price / 1_000_000
    return (round(input_cost + answer_cost, 5), round(input_cost + answer_cost + thinking, 5))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two expressions.

```python
def thinking_cost(inp, think, out, in_price, out_price):
    base = (inp * in_price + out * out_price) / 1e6
    return (round(base, 5), round(base + think * out_price / 1e6, 5))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute each part step by step. That costs O(1) time and O(1) space."

"I can do better: Two expressions. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/what-does-thinking-cost)

<a id="q2"></a>
### Q2. How Long Will the User Wait?

🟢 Warm-up · *Latency*

Time = time to first token `ttft` + (thinking + answer tokens) ÷ `speed` tokens per second. Return seconds (rounded to 1) for each thinking amount in `thinks`.

**Examples**

- `ttft = 0.6, speed = 80, answer = 300, thinks = [0, 2000, 16000]` → `[4.3, 29.4, 204.3]`

In [ ]:
def wait_time(ttft, speed, answer, thinks):
    """
    Time = time to first token ttft + (thinking + answer tokens) ÷ speed tokens
    per second. Return seconds (rounded to 1) for each thinking amount in
    thinks.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(wait_time, [
    ((0.6, 80, 300, [0, 2000, 16000]), [4.3, 29.4, 204.3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop.

```python
def wait_time(ttft, speed, answer, thinks):
    out = []
    for t in thinks:
        out.append(round(ttft + (t + answer) / speed, 1))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def wait_time(ttft, speed, answer, thinks):
    return [round(ttft + (t + answer) / speed, 1) for t in thinks]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(n) space."

"I can do better: A comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/how-long-will-the-user-wait)

<a id="q3"></a>
### Q3. How Many Samples for Best-of-N?

🟢 Warm-up · *Verifiers*

With a perfect verifier you only need one correct sample. If each is right with probability `p`, return the smallest n with P(at least one right) ≥ `target`.

**Examples**

- `p = 0.3, target = 0.9` → `7`
- `p = 0.05, target = 0.95` → `59`
- `p = 0.9, target = 0.9` → `1`

In [ ]:
def samples_needed(p, target):
    """
    With a perfect verifier you only need one correct sample. If each is right
    with probability p, return the smallest n with P(at least one right) ≥
    target.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(samples_needed, [
    ((0.3, 0.9), 7),
    ((0.05, 0.95), 59),
    ((0.9, 0.9), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Increase n until 1 − (1 − p)ⁿ reaches the target.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Solve with logarithms: n = ⌈log(1 − target) / log(1 − p)⌉.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Increase n until 1 − (1 − p)ⁿ reaches the target.

```python
def samples_needed(p, target):
    n = 1
    while 1 - (1 - p) ** n < target - 1e-12:
        n += 1
    return n
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Solve with logarithms: n = ⌈log(1 − target) / log(1 − p)⌉.

```python
def samples_needed(p, target):
    return max(1, math.ceil(math.log(1 - target) / math.log(1 - p) - 1e-9))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Increase n until 1 − (1 − p)ⁿ reaches the target. That costs O(n) time and O(1) space."

"I can do better: Solve with logarithms: n = ⌈log(1 − target) / log(1 − p)⌉. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/how-many-samples-for-best-of-n)

<a id="q4"></a>
### Q4. Find the Final Answer

🟢 Warm-up · *Parsing reasoning*

Return the value after the **last** 'Answer:' (case-insensitive) in the text, stripped, or None.

**Examples**

- `text = "Let's think. 12*3=36, minus 4... Answer: 30? No wait.\nAnswer: 32"` → `'32'`
- `text = "I'm not sure."` → `None`

In [ ]:
def final_answer(text):
    """
    Return the value after the last 'Answer:' (case-insensitive) in the text,
    stripped, or None.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(final_answer, [
    (("Let's think. 12*3=36, minus 4... Answer: 30? No wait.\nAnswer: 32",), '32'),
    (("I'm not sure.",), None),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan lines from the end.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`re.findall` and take the last match.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan lines from the end.

```python
def final_answer(text):
    lower = text.lower()
    i = lower.rfind("answer:")
    if i == -1:
        return None
    return text[i + len("answer:"):].split("\n")[0].strip()
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `re.findall` and take the last match.

```python
def final_answer(text):
    found = re.findall(r"answer:\s*(.*)", text, re.I)
    return found[-1].strip() if found else None
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan lines from the end. That costs O(n) time and O(1) space."

"I can do better: `re.findall` and take the last match. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/find-the-final-answer)

<a id="q5"></a>
### Q5. Separate Thinking From the Answer

🟢 Warm-up · *Response blocks*

A response's content is a list of blocks: `{'type': 'thinking', ...}`, `{'type': 'text', 'text': ...}` or `{'type': 'tool_use', ...}`. Return `(the joined text of all text blocks, number of thinking blocks)`. Show users the text, keep thinking for debugging.

**Examples**

- `blocks = [{'type': 'thinking', 'thinking': '...'}, {'type': 'text', 'text': 'The answer'}, {'type': 'text', 'text': ' is 42.'}]` → `('The answer is 42.', 1)`

In [ ]:
def split_blocks(blocks):
    """
    A response's content is a list of blocks: {'type': 'thinking', ...},
    {'type': 'text', 'text': ...} or {'type': 'tool_use', ...}. Return (the
    joined text of all text blocks, number of thinking blocks). Show users the
    text, keep thinking for debugging.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(split_blocks, [
    (([{'type': 'thinking', 'thinking': '...'}, {'type': 'text', 'text': 'The answer'}, {'type': 'text', 'text': ' is 42.'}],), ('The answer is 42.', 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A join over a filtered generator and a `sum`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with if-statements.

```python
def split_blocks(blocks):
    text, thinking = "", 0
    for b in blocks:
        if b["type"] == "text":
            text += b["text"]
        elif b["type"] == "thinking":
            thinking += 1
    return (text, thinking)
```

**Cost:** O(b) time, O(b) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A join over a filtered generator and a `sum`.

```python
def split_blocks(blocks):
    return ("".join(b["text"] for b in blocks if b["type"] == "text"), sum(b["type"] == "thinking" for b in blocks))
```

**Cost:** O(b) time, O(b) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with if-statements. That costs O(b) time and O(b) space."

"I can do better: A join over a filtered generator and a `sum`. That's O(b) time and O(b) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/separate-thinking-from-the-answer)

<a id="q6"></a>
### Q6. Choose the Effort Level

🟢 Warm-up · *Cheapest level that's good enough*

`EFFORT[level] = (accuracy, cost per task)` from your own evaluation. Return the cheapest level whose accuracy reaches `target`, or 'none'.

**Examples**

- `target = 0.8` → `'medium'`
- `target = 0.88` → `'high'`
- `target = 0.95` → `'none'`

In [ ]:
EFFORT = {"low": (0.71, 0.004), "medium": (0.83, 0.011), "high": (0.88, 0.032), "max": (0.89, 0.09)}

def choose_effort(target):
    """
    EFFORT[level] = (accuracy, cost per task) from your own evaluation. Return
    the cheapest level whose accuracy reaches target, or 'none'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(choose_effort, [
    ((0.8,), 'medium'),
    ((0.88,), 'high'),
    ((0.95,), 'none'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and keep the cheapest passing level.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`min` over a filtered generator with a default.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop and keep the cheapest passing level.

```python
def choose_effort(target):
    best = None
    for level, (acc, cost) in EFFORT.items():
        if acc >= target and (best is None or cost < EFFORT[best][1]):
            best = level
    return best if best else "none"
```

**Cost:** O(l) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `min` over a filtered generator with a default.

```python
def choose_effort(target):
    return min((l for l, (a, _) in EFFORT.items() if a >= target), key=lambda l: EFFORT[l][1], default="none")
```

**Cost:** O(l) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and keep the cheapest passing level. That costs O(l) time and O(1) space."

"I can do better: `min` over a filtered generator with a default. That's O(l) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/choose-the-effort-level)

<a id="q7"></a>
### Q7. Combine Numeric Estimates

🟢 Warm-up · *Median beats mean*

Several samples estimated a number; one went wild. Return `(mean, median)` rounded to 2 — the median ignores the outlier.

**Examples**

- `estimates = [42, 40, 44, 41, 4100]` → `(853.4, 42)`
- `estimates = [10, 12]` → `(11, 11.0)`

In [ ]:
import statistics

def combine(estimates):
    """
    Several samples estimated a number; one went wild. Return (mean, median)
    rounded to 2 — the median ignores the outlier.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(combine, [
    (([42, 40, 44, 41, 4100],), (853.4, 42)),
    (([10, 12],), (11, 11.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort and pick the middle by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`statistics.mean` and `statistics.median`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sort and pick the middle by hand.

```python
def combine(estimates):
    s = sorted(estimates)
    n = len(s)
    median = s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2
    return (round(sum(s) / n, 2), round(median, 2))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `statistics.mean` and `statistics.median`.

```python
def combine(estimates):
    return (round(statistics.mean(estimates), 2), round(statistics.median(estimates), 2))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort and pick the middle by hand. That costs O(n log n) time and O(n) space."

"I can do better: `statistics.mean` and `statistics.median`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/combine-numeric-estimates)

<a id="q8"></a>
### Q8. Accuracy Per Dollar

🟢 Warm-up · *Is reasoning worth it here?*

For each option `(name, accuracy, cost per task)`, compute correct answers per dollar and return the names sorted from best to worst value.

**Examples**

- `options = [('fast', 0.78, 0.002), ('fast+vote5', 0.86, 0.01), ('reasoning', 0.92, 0.03)]` → `['fast', 'fast+vote5', 'reasoning']`

In [ ]:
def value_rank(options):
    """
    For each option (name, accuracy, cost per task), compute correct answers per
    dollar and return the names sorted from best to worst value.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(value_rank, [
    (([('fast', 0.78, 0.002), ('fast+vote5', 0.86, 0.01), ('reasoning', 0.92, 0.03)],), ['fast', 'fast+vote5', 'reasoning']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute ratios in a loop, then sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sorted` with a key.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute ratios in a loop, then sort.

```python
def value_rank(options):
    scored = []
    for name, acc, cost in options:
        scored.append((acc / cost, name))
    scored.sort(reverse=True)
    return [n for _, n in scored]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `sorted` with a key.

```python
def value_rank(options):
    return [n for n, a, c in sorted(options, key=lambda o: o[1] / o[2], reverse=True)]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute ratios in a loop, then sort. That costs O(n log n) time and O(n) space."

"I can do better: `sorted` with a key. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/accuracy-per-dollar)

<a id="q9"></a>
### Q9. Stop Thinking at the Budget

🟢 Warm-up · *Budget control*

A model thinks in chunks of tokens. Stop adding chunks once the next one would exceed `budget`. Return `(chunks used, tokens used, chunks cut)`.

**Examples**

- `chunks = [1200, 3000, 800, 4000, 500], budget = 5000` → `(3, 5000, 2)`

In [ ]:
def think_budget(chunks, budget):
    """
    A model thinks in chunks of tokens. Stop adding chunks once the next one
    would exceed budget. Return (chunks used, tokens used, chunks cut).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(think_budget, [
    (([1200, 3000, 800, 4000, 500], 5000), (3, 5000, 2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A running total that breaks at the limit.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.accumulate` to find where the total crosses the budget.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A running total that breaks at the limit.

```python
def think_budget(chunks, budget):
    used, n = 0, 0
    for c in chunks:
        if used + c > budget:
            break
        used += c
        n += 1
    return (n, used, len(chunks) - n)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.accumulate` to find where the total crosses the budget.

```python
def think_budget(chunks, budget):
    from itertools import accumulate, takewhile
    totals = list(takewhile(lambda t: t <= budget, accumulate(chunks)))
    return (len(totals), totals[-1] if totals else 0, len(chunks) - len(totals))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A running total that breaks at the limit. That costs O(n) time and O(1) space."

"I can do better: `itertools.accumulate` to find where the total crosses the budget. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/stop-thinking-at-the-budget)

<a id="q10"></a>
### Q10. Prompt for Reasoning Without a Reasoning Model

🟢 Warm-up · *Chain-of-thought prompts*

Build a prompt that asks a standard model to reason first: the question, then 'Think step by step inside &lt;thinking&gt; tags, then give only the final answer inside &lt;answer&gt; tags.' Then parse a reply: return the text inside the last &lt;answer&gt;…&lt;/answer&gt;, stripped. Return `(prompt, parsed answer)`.

**Examples**

- `question = 'What is 17 × 6?', reply = '<thinking>17×6 = 102</thinking><answer> 102 </answer>'` → `('What is 17 × 6?\n\nThink step by step inside <thinking> tags, then give only the final answer inside <answer> tags.', '102')`

In [ ]:
def cot(question, reply):
    """
    Build a prompt that asks a standard model to reason first: the question,
    then 'Think step by step inside <thinking> tags, then give only the final
    answer inside <answer> tags.' Then parse a reply: return the text inside the
    last <answer>…</answer>, stripped. Return (prompt, parsed answer).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cot, [
    (('What is 17 × 6?', '<thinking>17×6 = 102</thinking><answer> 102 </answer>'), ('What is 17 × 6?\n\nThink step by step inside <thinking> tags, then give only the final answer inside <answer> tags.', '102')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

String concatenation and `find`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

An f-string and a regex.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** String concatenation and `find`.

```python
def cot(question, reply):
    prompt = question + "\n\nThink step by step inside <thinking> tags, then give only the final answer inside <answer> tags."
    start = reply.rfind("<answer>")
    end = reply.rfind("</answer>")
    answer = reply[start + len("<answer>"):end].strip() if start != -1 and end > start else None
    return (prompt, answer)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** An f-string and a regex.

```python
def cot(question, reply):
    prompt = f"{question}\n\nThink step by step inside <thinking> tags, then give only the final answer inside <answer> tags."
    found = re.findall(r"<answer>(.*?)</answer>", reply, re.S)
    return (prompt, found[-1].strip() if found else None)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: String concatenation and `find`. That costs O(n) time and O(n) space."

"I can do better: An f-string and a regex. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/prompt-for-reasoning-without-a-reasoning-model)

<a id="q11"></a>
### Q11. Majority Voting Helps — How Much?

🟡 Interview standard · *Self-consistency*

If each sample is right with probability `p` independently, what is the chance that a strict majority of `n` samples (n odd) is right? Round to 4.

**Examples**

- `p = 0.7, n = 1` → `0.7`
- `p = 0.7, n = 5` → `0.8369`
- `p = 0.7, n = 15` → `0.95`

In [ ]:
def majority_right(p, n):
    """
    If each sample is right with probability p independently, what is the chance
    that a strict majority of n samples (n odd) is right? Round to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(majority_right, [
    ((0.7, 1), 0.7),
    ((0.7, 5), 0.8369),
    ((0.7, 15), 0.95),
    ((0.4, 15), 0.2131),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Enumerate every right/wrong pattern of n samples.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Add binomial probabilities with `math.comb`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Enumerate every right/wrong pattern of n samples.

```python
def majority_right(p, n):
    total = 0.0
    for pattern in product([0, 1], repeat=n):
        k = sum(pattern)
        if k > n / 2:
            total += p ** k * (1 - p) ** (n - k)
    return round(total, 4)
```

**Cost:** O(2ⁿ · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Add binomial probabilities with `math.comb`.

```python
def majority_right(p, n):
    return round(sum(math.comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n // 2 + 1, n + 1)), 4)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Enumerate every right/wrong pattern of n samples. That costs O(2ⁿ · n) time and O(n) space."

"I can do better: Add binomial probabilities with `math.comb`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/majority-voting-helps-how-much)

<a id="q12"></a>
### Q12. Unbiased pass@k

🟡 Interview standard · *Evaluating code models*

From `n` samples of which `c` passed the tests, estimate pass@k = the chance at least one of k random samples passes: 1 − C(n−c, k) / C(n, k). Round to 4.

**Examples**

- `n = 20, c = 3, k = 1` → `0.15`
- `n = 20, c = 3, k = 5` → `0.6009`
- `n = 10, c = 0, k = 3` → `0.0`

In [ ]:
def pass_at_k(n, c, k):
    """
    From n samples of which c passed the tests, estimate pass@k = the chance at
    least one of k random samples passes: 1 − C(n−c, k) / C(n, k). Round to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pass_at_k, [
    ((20, 3, 1), 0.15),
    ((20, 3, 5), 0.6009),
    ((10, 0, 3), 0.0),
    ((10, 9, 3), 1.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A running product: Π (1 − k / i) for i from n−c+1 to n.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.comb` directly.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A running product: Π (1 − k / i) for i from n−c+1 to n.

```python
def pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    prod = 1.0
    for i in range(n - c + 1, n + 1):
        prod *= 1 - k / i
    return round(1 - prod, 4)
```

**Cost:** O(c) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `math.comb` directly.

```python
def pass_at_k(n, c, k):
    return round(1 - math.comb(n - c, k) / math.comb(n, k), 4) if n - c >= k else 1.0
```

**Cost:** O(k) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A running product: Π (1 − k / i) for i from n−c+1 to n. That costs O(c) time and O(1) space."

"I can do better: `math.comb` directly. That's O(k) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/unbiased-pass-k)

<a id="q13"></a>
### Q13. Route Easy and Hard Questions

🟡 Interview standard · *Reasoning only where it pays*

Send a question to the reasoning model if its difficulty score ≥ `cutoff`, else to the fast model. Accuracy and cost per question: fast (0.95 easy, 0.40 hard, \$0.002), reasoning (0.97 easy, 0.85 hard, \$0.03). Difficulty ≥ 0.5 means the question is really hard. Return `(expected accuracy, total cost)` rounded to 3 and 4.

**Examples**

- `difficulty = [0.1, 0.2, 0.9, 0.3, 0.7, 0.1, 0.8, 0.2], cutoff = 0.5` → `(0.912, 0.1)`
- `difficulty = [0.1, 0.2, 0.9, 0.3, 0.7, 0.1, 0.8, 0.2], cutoff = 0.0` → `(0.925, 0.24)`
- `difficulty = [0.1, 0.2, 0.9, 0.3, 0.7, 0.1, 0.8, 0.2], cutoff = 1.1` → `(0.744, 0.016)`

In [ ]:
def route_cost(difficulty, cutoff):
    """
    Send a question to the reasoning model if its difficulty score ≥ cutoff,
    else to the fast model. Accuracy and cost per question: fast (0.95 easy,
    0.40 hard, $0.002), reasoning (0.97 easy, 0.85 hard, $0.03). Difficulty ≥
    0.5 means the question is really hard. Return (expected accuracy, total
    cost) rounded to 3 and 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(route_cost, [
    (([0.1, 0.2, 0.9, 0.3, 0.7, 0.1, 0.8, 0.2], 0.5), (0.912, 0.1)),
    (([0.1, 0.2, 0.9, 0.3, 0.7, 0.1, 0.8, 0.2], 0.0), (0.925, 0.24)),
    (([0.1, 0.2, 0.9, 0.3, 0.7, 0.1, 0.8, 0.2], 1.1), (0.744, 0.016)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and add up.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A table of (model, hard?) → accuracy and two sums.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop and add up.

```python
def route_cost(difficulty, cutoff):
    acc, cost = 0.0, 0.0
    for d in difficulty:
        hard = d >= 0.5
        if d >= cutoff:
            acc += 0.85 if hard else 0.97
            cost += 0.03
        else:
            acc += 0.40 if hard else 0.95
            cost += 0.002
    return (round(acc / len(difficulty), 3), round(cost, 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A table of (model, hard?) → accuracy and two sums.

```python
def route_cost(difficulty, cutoff):
    table = {("reason", True): (0.85, 0.03), ("reason", False): (0.97, 0.03),
             ("fast", True): (0.40, 0.002), ("fast", False): (0.95, 0.002)}
    picks = [table[("reason" if d >= cutoff else "fast", d >= 0.5)] for d in difficulty]
    return (round(sum(a for a, _ in picks) / len(picks), 3), round(sum(c for _, c in picks), 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and add up. That costs O(n) time and O(1) space."

"I can do better: A table of (model, hard?) → accuracy and two sums. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/route-easy-and-hard-questions)

<a id="q14"></a>
### Q14. Judge the Steps, Not Just the Answer

🟡 Interview standard · *Process vs outcome scoring*

Each candidate solution has per-step verifier scores and a final-answer score. Outcome scoring picks the best final score; process scoring picks the best **weakest step** (a chain is only as strong as its worst step). Return `(outcome pick, process pick)` — indexes, ties to the earlier.

**Examples**

- `cands = [([0.9, 0.2, 0.95], 0.97), ([0.8, 0.85, 0.8], 0.9), ([0.99, 0.99, 0.5], 0.92)]` → `(0, 1)`

In [ ]:
def pick_solutions(cands):
    """
    Each candidate solution has per-step verifier scores and a final-answer
    score. Outcome scoring picks the best final score; process scoring picks the
    best weakest step (a chain is only as strong as its worst step). Return
    (outcome pick, process pick) — indexes, ties to the earlier.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pick_solutions, [
    (([([0.9, 0.2, 0.95], 0.97), ([0.8, 0.85, 0.8], 0.9), ([0.99, 0.99, 0.5], 0.92)],), (0, 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two loops tracking the best.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` with keys over the indexes.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Two loops tracking the best.

```python
def pick_solutions(cands):
    best_o, best_p = 0, 0
    for i, (steps, final) in enumerate(cands):
        if final > cands[best_o][1]:
            best_o = i
        if min(steps) > min(cands[best_p][0]):
            best_p = i
    return (best_o, best_p)
```

**Cost:** O(c · s) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max` with keys over the indexes.

```python
def pick_solutions(cands):
    idx = range(len(cands))
    return (max(idx, key=lambda i: cands[i][1]), max(idx, key=lambda i: min(cands[i][0])))
```

**Cost:** O(c · s) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two loops tracking the best. That costs O(c · s) time and O(1) space."

"I can do better: `max` with keys over the indexes. That's O(c · s) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/judge-the-steps-not-just-the-answer)

<a id="q15"></a>
### Q15. Check the Arithmetic in the Working

🟡 Interview standard · *Verifying reasoning*

Find lines of the form `a op b = c` (integers, op in + − × \* /) and return the 1-based line numbers whose result is wrong. '×' means multiply. Division must be exact.

**Examples**

- `text = 'We have 12 boxes.\n12 * 8 = 96\n96 + 15 = 101\n101 - 7 = 94\n94 / 2 = 47\n3 × 4 = 13'` → `[3, 6]`

In [ ]:
def wrong_lines(text):
    """
    Find lines of the form a op b = c (integers, op in + − × * /) and return the
    1-based line numbers whose result is wrong. '×' means multiply. Division
    must be exact.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(wrong_lines, [
    (('We have 12 boxes.\n12 * 8 = 96\n96 + 15 = 101\n101 - 7 = 94\n94 / 2 = 47\n3 × 4 = 13',), [3, 6]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split each line by hand and compute.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One regex with groups and an operator table.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Split each line by hand and compute.

```python
def wrong_lines(text):
    bad = []
    for i, line in enumerate(text.split("\n"), 1):
        parts = line.replace("×", "*").split()
        if len(parts) == 5 and parts[3] == "=" and parts[1] in "+-*/" and all(p.lstrip("-").isdigit() for p in (parts[0], parts[2], parts[4])):
            a, op, b, c = int(parts[0]), parts[1], int(parts[2]), int(parts[4])
            if op == "+":
                v = a + b
            elif op == "-":
                v = a - b
            elif op == "*":
                v = a * b
            else:
                v = a / b
            if v != c:
                bad.append(i)
    return bad
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One regex with groups and an operator table.

```python
def wrong_lines(text):
    ops = {"+": lambda a, b: a + b, "-": lambda a, b: a - b, "*": lambda a, b: a * b, "×": lambda a, b: a * b,
           "/": lambda a, b: a / b}
    bad = []
    for i, line in enumerate(text.splitlines(), 1):
        m = re.fullmatch(r"\s*(-?\d+)\s*([-+*/×])\s*(-?\d+)\s*=\s*(-?\d+)\s*", line)
        if m and ops[m[2]](int(m[1]), int(m[3])) != int(m[4]):
            bad.append(i)
    return bad
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split each line by hand and compute. That costs O(n) time and O(1) space."

"I can do better: One regex with groups and an operator table. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/check-the-arithmetic-in-the-working)

<a id="q16"></a>
### Q16. Know When to Abstain

🟡 Interview standard · *Agreement as confidence*

Sample answers several times. If the most common answer has at least `threshold` of the votes, return it; otherwise return 'not sure — escalate'. In high-stakes settings, saying 'I don't know' beats a confident guess.

**Examples**

- `samples = ['B', 'B', 'B', 'A', 'B'], threshold = 0.6` → `'B'`
- `samples = ['A', 'B', 'C', 'A', 'B'], threshold = 0.6` → `'not sure — escalate'`

In [ ]:
def answer_or_abstain(samples, threshold):
    """
    Sample answers several times. If the most common answer has at least
    threshold of the votes, return it; otherwise return 'not sure — escalate'.
    In high-stakes settings, saying 'I don't know' beats a confident guess.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(answer_or_abstain, [
    ((['B', 'B', 'B', 'A', 'B'], 0.6), 'B'),
    ((['A', 'B', 'C', 'A', 'B'], 0.6), 'not sure — escalate'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count with a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter.most_common(1)`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count with a dict.

```python
def answer_or_abstain(samples, threshold):
    counts = {}
    for s in samples:
        counts[s] = counts.get(s, 0) + 1
    best = max(counts, key=counts.get)
    if counts[best] / len(samples) >= threshold:
        return best
    return "not sure — escalate"
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter.most_common(1)`.

```python
def answer_or_abstain(samples, threshold):
    best, n = Counter(samples).most_common(1)[0]
    return best if n / len(samples) >= threshold else "not sure — escalate"
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count with a dict. That costs O(n) time and O(n) space."

"I can do better: `Counter.most_common(1)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/know-when-to-abstain)

<a id="q17"></a>
### Q17. Sampling Many Times, Cheaply

🟡 Interview standard · *Caching the shared prompt*

Run `n` samples of the same prompt (`inp` tokens), each writing `out` tokens. With caching, the first sample writes the prompt to the cache (1.25×) and the rest read it (0.1×). Prices per million: input `pin`, output `pout`. Return `(cost without caching, with caching)` rounded to 4.

**Examples**

- `n = 5, inp = 20000, out = 800, pin = 3.0, pout = 15.0` → `(0.36, 0.159)`

In [ ]:
def sampling_cost(n, inp, out, pin, pout):
    """
    Run n samples of the same prompt (inp tokens), each writing out tokens. With
    caching, the first sample writes the prompt to the cache (1.25×) and the
    rest read it (0.1×). Prices per million: input pin, output pout. Return
    (cost without caching, with caching) rounded to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sampling_cost, [
    ((5, 20000, 800, 3.0, 15.0), (0.36, 0.159)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add sample by sample.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A closed form.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Add sample by sample.

```python
def sampling_cost(n, inp, out, pin, pout):
    plain = cached = 0.0
    for i in range(n):
        plain += (inp * pin + out * pout) / 1e6
        factor = 1.25 if i == 0 else 0.1
        cached += (inp * pin * factor + out * pout) / 1e6
    return (round(plain, 4), round(cached, 4))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A closed form.

```python
def sampling_cost(n, inp, out, pin, pout):
    outputs = n * out * pout / 1e6
    return (round(n * inp * pin / 1e6 + outputs, 4), round((1.25 + 0.1 * (n - 1)) * inp * pin / 1e6 + outputs, 4))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add sample by sample. That costs O(n) time and O(1) space."

"I can do better: A closed form. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/sampling-many-times-cheaply)

<a id="q18"></a>
### Q18. Does More Thinking Pay Off?

🟡 Interview standard · *Diminishing returns*

Given evaluation results `(thinking budget, accuracy)` sorted by budget, return the largest budget where the accuracy gained per extra 1,000 tokens (vs the previous budget) is still at least `min_gain` percentage points.

**Examples**

- `results = [(0, 61.0), (1000, 70.0), (4000, 78.0), (16000, 81.0), (32000, 81.5)], min_gain = 0.5` → `4000`
- `results = [(0, 61.0), (1000, 61.2)], min_gain = 0.5` → `0`

In [ ]:
def worth_it(results, min_gain):
    """
    Given evaluation results (thinking budget, accuracy) sorted by budget,
    return the largest budget where the accuracy gained per extra 1,000 tokens
    (vs the previous budget) is still at least min_gain percentage points.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(worth_it, [
    (([(0, 61.0), (1000, 70.0), (4000, 78.0), (16000, 81.0), (32000, 81.5)], 0.5), 4000),
    (([(0, 61.0), (1000, 61.2)], 0.5), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk pairs of neighbours.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`zip(results, results[1:])` and keep the last that passes.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk pairs of neighbours.

```python
def worth_it(results, min_gain):
    best = results[0][0]
    for i in range(1, len(results)):
        b0, a0 = results[i - 1]
        b1, a1 = results[i]
        if (a1 - a0) / ((b1 - b0) / 1000) >= min_gain:
            best = b1
        else:
            break
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `zip(results, results[1:])` and keep the last that passes.

```python
def worth_it(results, min_gain):
    best = results[0][0]
    for (b0, a0), (b1, a1) in zip(results, results[1:]):
        if (a1 - a0) * 1000 / (b1 - b0) < min_gain:
            break
        best = b1
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk pairs of neighbours. That costs O(n) time and O(1) space."

"I can do better: `zip(results, results[1:])` and keep the last that passes. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/does-more-thinking-pay-off)

<a id="q19"></a>
### Q19. Normalise Before You Vote

🟡 Interview standard · *Free-form answers*

'\$32', '32 dollars', '32.0' and 'thirty-two' are the same answer. Normalise each answer to a number (strip symbols and words, parse number words up to 'fifty-nine' written with a hyphen), then return the majority value as an int and its vote count. Unparseable answers are ignored.

**Examples**

- `answers = ['$32', '32 dollars', 'thirty-two', '32.0', '31', 'no idea']` → `(32, 4)`

In [ ]:
WORDS = {"zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
         "ten": 10, "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50}

def normalised_vote(answers):
    """
    '$32', '32 dollars', '32.0' and 'thirty-two' are the same answer. Normalise
    each answer to a number (strip symbols and words, parse number words up to
    'fifty-nine' written with a hyphen), then return the majority value as an
    int and its vote count. Unparseable answers are ignored.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(normalised_vote, [
    ((['$32', '32 dollars', 'thirty-two', '32.0', '31', 'no idea'],), (32, 4)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Character-by-character cleaning and a word table.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regex for the first number, the word table for the rest, then `Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Character-by-character cleaning and a word table.

```python
def normalised_vote(answers):
    counts = {}
    for a in answers:
        a = a.lower().strip()
        digits = ""
        for ch in a:
            if ch.isdigit() or (ch == "." and digits):
                digits += ch
            elif digits:
                break
        value = None
        if digits:
            value = int(float(digits))
        else:
            parts = a.split("-")
            if all(p in WORDS for p in parts):
                value = sum(WORDS[p] for p in parts)
        if value is not None:
            counts[value] = counts.get(value, 0) + 1
    best = max(counts, key=counts.get)
    return (best, counts[best])
```

**Cost:** O(n · l) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regex for the first number, the word table for the rest, then `Counter`.

```python
def normalised_vote(answers):
    def parse(a):
        a = a.lower().strip()
        m = re.search(r"\d+(?:\.\d+)?", a)
        if m:
            return int(float(m.group()))
        parts = a.split("-")
        return sum(WORDS[p] for p in parts) if all(p in WORDS for p in parts) else None
    values = [v for v in map(parse, answers) if v is not None]
    return Counter(values).most_common(1)[0]
```

**Cost:** O(n · l) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Character-by-character cleaning and a word table. That costs O(n · l) time and O(n) space."

"I can do better: A regex for the first number, the word table for the rest, then `Counter`. That's O(n · l) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `answers = ['$32', '32 dollars', 'thirty-two', '31']`

1. parse() returns 32
2. parse() returns 32
3. parse() returns 32
4. parse() returns 31
5. normalised\_vote() returns (32, 3)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/normalise-before-you-vote)

<a id="q20"></a>
### Q20. Beam Search Over Reasoning Steps

🔴 Stretch · *Search with a verifier*

At each step there are candidate next steps with verifier scores. Keep the best `beam` partial solutions (scored by the sum of step scores) and extend each with every candidate. `steps[i]` is the list of (label, score) options at step i, the same for every path. Return the best full path (labels) and its score rounded to 2.

**Examples**

- `steps = [[('a', 0.9), ('b', 0.6)], [('c', 0.2), ('d', 0.8)], [('e', 0.5), ('f', 0.95)]], beam = 2` → `(['a', 'd', 'f'], 2.65)`

In [ ]:
def beam_path(steps, beam):
    """
    At each step there are candidate next steps with verifier scores. Keep the
    best beam partial solutions (scored by the sum of step scores) and extend
    each with every candidate. steps[i] is the list of (label, score) options at
    step i, the same for every path. Return the best full path (labels) and its
    score rounded to 2.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(beam_path, [
    (([[('a', 0.9), ('b', 0.6)], [('c', 0.2), ('d', 0.8)], [('e', 0.5), ('f', 0.95)]], 2), (['a', 'd', 'f'], 2.65)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Enumerate every path (for this small case) and take the best.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep only the top-b partial paths with `heapq.nlargest` at each step.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Enumerate every path (for this small case) and take the best.

```python
def beam_path(steps, beam):
    best = None
    for path in product(*steps):
        score = sum(s for _, s in path)
        if best is None or score > best[1]:
            best = ([l for l, _ in path], score)
    return (best[0], round(best[1], 2))
```

**Cost:** O(kˢ) time, O(kˢ) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Keep only the top-b partial paths with `heapq.nlargest` at each step.

```python
def beam_path(steps, beam):
    import heapq
    beams = [([], 0.0)]
    for options in steps:
        grown = [(path + [l], score + s) for path, score in beams for l, s in options]
        beams = heapq.nlargest(beam, grown, key=lambda b: b[1])
    path, score = beams[0]
    return (path, round(score, 2))
```

**Cost:** O(s · b · k log b) time, O(b) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Enumerate every path (for this small case) and take the best. That costs O(kˢ) time and O(kˢ) space."

"I can do better: Keep only the top-b partial paths with `heapq.nlargest` at each step. That's O(s · b · k log b) time and O(b) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/reasoning/beam-search-over-reasoning-steps)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why are reasoning models expensive?</b></summary>

They generate many thinking tokens before the answer, and those are billed as output tokens, which cost more than input. Latency also grows with the thinking length.

</details>

<details>
<summary><b>2. What is self-consistency?</b></summary>

Sampling several answers and taking the most common. It improves accuracy when each sample is more likely right than wrong, and agreement doubles as a confidence signal.

</details>

<details>
<summary><b>3. Best-of-N vs majority vote?</b></summary>

Majority vote needs no verifier but needs most samples to agree. Best-of-N needs a verifier but only needs one correct sample, so it scales better when a reliable check exists.

</details>

<details>
<summary><b>4. What is pass@k and why the special formula?</b></summary>

The chance that at least one of k samples passes the tests. Estimating it from n ≥ k samples with 1 − C(n−c,k)/C(n,k) is unbiased and lower-variance than drawing exactly k samples.

</details>

<details>
<summary><b>5. How do you decide whether to use a reasoning model?</b></summary>

Evaluate on your own tasks at different effort levels and models, then choose by accuracy per dollar and latency. Route only hard requests to reasoning.

</details>

<details>
<summary><b>6. Process vs outcome reward?</b></summary>

Outcome rewards score only the final answer; process rewards score each step. Process signals catch right answers reached by wrong reasoning and help search.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: What a reasoning model does differently
- [ ] I can explain: Choosing how much to think
- [ ] I can explain: Sampling more: self-consistency and best-of-N
- [ ] I can explain: Evaluating reasoning: pass@k and checking the working
- [ ] I can explain: Search over reasoning

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Know When to Abstain
- [ ] [Q17](#q17) Sampling Many Times, Cheaply
- [ ] [Q18](#q18) Does More Thinking Pay Off?
- [ ] [Q19](#q19) Normalise Before You Vote
- [ ] [Q20](#q20) Beam Search Over Reasoning Steps

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.